# Teste A/B do início ao fim: dados reais do GA4, experimento simulado

**Objetivo:** praticar o ciclo completo de um teste A/B (planejamento, validação, análise e decisão) usando o comportamento real dos usuários da Google Merchandise Store (export público do GA4 no BigQuery).

**Honestidade metodológica:** o dataset não tem nenhum experimento. Os usuários e as conversões são reais, mas a divisão em grupos A e B e o efeito do "tratamento" são **simulados**, com um efeito verdadeiro conhecido. Isso permite checar se a análise recupera a verdade, coisa impossível num teste real.

Roteiro:
1. Linha de base (taxa de conversão, receita por usuário)
2. Planejamento: tamanho de amostra e duração
3. Teste A/A: o método erra só 5% das vezes?
4. Teste A/B simulado: SRM, conversão, receita
5. Por que não olhar o resultado todo dia (*peeking*)
6. Decisão

In [ ]:
# ==== CONFIGURAÇÃO ====
PROJECT_ID = 'seu-projeto-gcp'   # <- troque pelo ID do seu projeto no BigQuery
USE_SYNTHETIC_DATA = False       # True = roda sem BigQuery, com dados inventados (só para testar o notebook)
TRUE_RELATIVE_LIFT = 0.10        # efeito verdadeiro injetado no grupo B (+10% na conversão)
SEED = 42

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, hashlib
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, confint_proportions_2indep
rng = np.random.default_rng(SEED)
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. Carregar os dados (um registro por usuário)

A query transforma eventos em uma tabela por usuário: primeiro dia visto, se comprou e quanto gastou. Repare no CTE: primeiro extraímos o `ga_session_id` com `UNNEST`, depois agregamos.

In [ ]:
QUERY = """
with events as (
  select
    user_pseudo_id,
    parse_date('%Y%m%d', event_date) as event_date,
    event_name,
    (select value.int_value from unnest(event_params) where key = 'ga_session_id') as ga_session_id,
    ecommerce.purchase_revenue_in_usd as revenue
  from `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  where _table_suffix between '20201101' and '20210131'
)
select
  user_pseudo_id,
  min(event_date)                                        as first_seen_date,
  count(distinct ga_session_id)                          as sessions,
  logical_or(event_name = 'purchase')                    as converted,
  sum(if(event_name = 'purchase', coalesce(revenue, 0), 0)) as revenue_usd
from events
group by user_pseudo_id
"""

if USE_SYNTHETIC_DATA:
    n = 270_000
    users = pd.DataFrame({'user_pseudo_id': [f'u{i}' for i in range(n)]})
    users['first_seen_date'] = pd.to_datetime('2020-11-01') + pd.to_timedelta(rng.integers(0, 92, n), unit='D')
    users['sessions'] = rng.geometric(0.6, n)
    users['converted'] = rng.random(n) < 0.015
    users['revenue_usd'] = np.where(users.converted, rng.lognormal(4.0, 0.8, n), 0.0)
else:
    from google.colab import auth
    auth.authenticate_user()
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT_ID)
    users = client.query(QUERY).to_dataframe()
    users['first_seen_date'] = pd.to_datetime(users['first_seen_date'])

users['converted'] = users['converted'].astype(bool)
print(f'{len(users):,} usuários')
users.head()

## 2. Linha de base

In [ ]:
p0 = users.converted.mean()
rpu = users.revenue_usd.mean()
aov = users.loc[users.converted, 'revenue_usd'].mean()
users_per_day = users.groupby('first_seen_date').size().mean()
print(f'Taxa de conversão (usuário): {p0:.3%}')
print(f'Receita por usuário:         US$ {rpu:,.2f}')
print(f'Ticket médio (compradores):  US$ {aov:,.2f}')
print(f'Novos usuários por dia:      {users_per_day:,.0f}')

## 3. Planejamento: de quantos usuários precisamos?

Antes de rodar qualquer teste, define-se o **efeito mínimo detectável (MDE)**, o nível de significância (α = 5%) e o poder (80%). Com a conversão baixa típica de e-commerce, detectar efeitos pequenos exige muita gente.

In [ ]:
power = NormalIndPower()
rows = []
for mde in [0.05, 0.10, 0.15, 0.20, 0.30]:
    es = proportion_effectsize(p0 * (1 + mde), p0)
    n_per_group = power.solve_power(effect_size=es, alpha=0.05, power=0.8, ratio=1, alternative='two-sided')
    rows.append({'MDE relativo': f'{mde:.0%}', 'usuários por grupo': int(np.ceil(n_per_group)),
                 'dias necessários (50/50)': np.ceil(2 * n_per_group / users_per_day)})
plan = pd.DataFrame(rows); plan

In [ ]:
mdes = np.linspace(0.05, 0.4, 30)
ns = [power.solve_power(proportion_effectsize(p0*(1+m), p0), alpha=0.05, power=0.8) for m in mdes]
plt.figure(figsize=(7,4)); plt.plot(mdes*100, ns); plt.yscale('log')
plt.xlabel('Efeito mínimo detectável (% relativo)'); plt.ylabel('Usuários por grupo (log)')
plt.title('Quanto menor o efeito, mais usuários: a relação é quadrática'); plt.grid(alpha=.3); plt.show()

**Para escrever com suas palavras:** com a conversão desta loja, quantos dias um teste para detectar +10% levaria? O que você diria a um time de produto que quer testar a cor de um botão em uma semana?

## 4. Teste A/A: o método é confiável?

Divide-se a base aleatoriamente em dois grupos **sem tratamento nenhum**, mil vezes. Se o teste estatístico estiver correto, cerca de 5% das divisões devem dar "significativo" (p < 0,05) por puro acaso, e os p-valores devem ter distribuição uniforme.

In [ ]:
conv = users.converted.to_numpy()
n_sims = 1000
pvals = np.empty(n_sims)
for i in range(n_sims):
    g = rng.random(len(conv)) < 0.5
    _, pvals[i] = proportions_ztest([conv[g].sum(), conv[~g].sum()], [g.sum(), (~g).sum()])
print(f'Taxa de falso positivo no A/A: {(pvals < 0.05).mean():.1%}  (esperado: ~5%)')
plt.figure(figsize=(7,3)); plt.hist(pvals, bins=20, edgecolor='white')
plt.title('p-valores do A/A: devem ser ~uniformes'); plt.xlabel('p-valor'); plt.show()

## 5. Teste A/B simulado

**Atribuição:** hash do `user_pseudo_id`, como fazem as ferramentas de experimentação. Um mesmo usuário cai sempre no mesmo grupo.

**Tratamento:** no grupo B, uma parte dos não compradores passa a comprar, de forma que a conversão suba exatamente `TRUE_RELATIVE_LIFT`. Os novos compradores recebem valores de compra sorteados entre as compras reais.

In [ ]:
def bucket(uid):
    return int(hashlib.md5(uid.encode()).hexdigest(), 16) % 2

ab = users.copy()
ab['group'] = np.where(ab.user_pseudo_id.map(bucket) == 1, 'B', 'A')

# injeta o efeito no grupo B
is_b = (ab.group == 'B').to_numpy()
not_conv_b = is_b & ~ab.converted.to_numpy()
flip_prob = p0 * TRUE_RELATIVE_LIFT / (1 - p0)
flip = not_conv_b & (rng.random(len(ab)) < flip_prob)
real_orders = ab.loc[ab.converted, 'revenue_usd'].to_numpy()
ab.loc[flip, 'converted'] = True
ab.loc[flip, 'revenue_usd'] = rng.choice(real_orders, flip.sum())
ab.groupby('group').agg(users=('user_pseudo_id', 'size'), conversion=('converted', 'mean'), revenue_per_user=('revenue_usd', 'mean'))

### 5.1 Checagem de SRM (Sample Ratio Mismatch)

**Antes de olhar qualquer métrica**, confirma-se que a divisão saiu 50/50. Um desvio significativo indica bug na atribuição ou no tracking, e aí o resultado não vale, por melhor que pareça.

In [ ]:
counts = ab.group.value_counts().reindex(['A', 'B']).to_numpy()
chi2, p_srm = stats.chisquare(counts, f_exp=[counts.sum() / 2] * 2)
print(f'A: {counts[0]:,}  B: {counts[1]:,}  |  p-valor SRM = {p_srm:.4f}')
print('OK: sem desvio de proporção' if p_srm > 0.01 else 'ALERTA: SRM, investigar antes de analisar')

### 5.2 Métrica principal: taxa de conversão

In [ ]:
a, b = ab[ab.group == 'A'], ab[ab.group == 'B']
x = np.array([b.converted.sum(), a.converted.sum()]); n = np.array([len(b), len(a)])
z, p_conv = proportions_ztest(x, n)
low, high = confint_proportions_2indep(x[0], n[0], x[1], n[1], method='wald')
pa, pb = a.converted.mean(), b.converted.mean()
print(f'Conversão A = {pa:.3%} | B = {pb:.3%} | lift relativo = {pb/pa - 1:+.1%}')
print(f'Diferença absoluta = {pb - pa:+.3%} (IC 95%: {low:+.3%} a {high:+.3%})')
print(f'p-valor = {p_conv:.4f}  ->  efeito verdadeiro injetado: {TRUE_RELATIVE_LIFT:+.0%}')

### 5.3 Métrica secundária: receita por usuário

Receita tem cauda longa (poucos pedidos enormes), então além do teste t de Welch usamos **bootstrap** para o intervalo de confiança, que não supõe normalidade.

In [ ]:
t, p_rev = stats.ttest_ind(b.revenue_usd, a.revenue_usd, equal_var=False)
ra, rb = a.revenue_usd.to_numpy(), b.revenue_usd.to_numpy()
boot = np.array([rng.choice(rb, len(rb)).mean() - rng.choice(ra, len(ra)).mean() for _ in range(1000)])
print(f'Receita/usuário A = US$ {ra.mean():.2f} | B = US$ {rb.mean():.2f}')
print(f'Diferença = US$ {rb.mean() - ra.mean():+.2f} | IC 95% bootstrap: US$ {np.percentile(boot, 2.5):+.2f} a {np.percentile(boot, 97.5):+.2f}')
print(f'p-valor (Welch) = {p_rev:.4f}')
plt.figure(figsize=(7,3)); plt.hist(boot, bins=40, edgecolor='white'); plt.axvline(0, color='k', ls='--')
plt.title('Distribuição bootstrap da diferença de receita por usuário'); plt.show()

## 6. Por que não "espiar" o teste todo dia (*peeking*)

Simulamos vários testes A/A acompanhados dia a dia. Se o time encerra o teste **no primeiro dia** em que p < 0,05, a taxa de falso positivo explode bem acima dos 5% prometidos.

In [ ]:
days = np.sort(users.first_seen_date.unique())[:30]
day_idx = users.first_seen_date.map({d: i for i, d in enumerate(days)}).to_numpy()
mask = ~np.isnan(day_idx.astype(float))
conv_d, day_d = conv[mask], day_idx[mask].astype(int)

def daily_pvalues():
    g = rng.random(len(conv_d)) < 0.5
    n_a = np.cumsum(np.bincount(day_d[~g], minlength=30)); x_a = np.cumsum(np.bincount(day_d[~g], weights=conv_d[~g], minlength=30))
    n_b = np.cumsum(np.bincount(day_d[g],  minlength=30)); x_b = np.cumsum(np.bincount(day_d[g],  weights=conv_d[g],  minlength=30))
    p_pool = (x_a + x_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
    zz = np.divide(x_b/n_b - x_a/n_a, se, out=np.zeros(30), where=se > 0)
    return 2 * stats.norm.sf(np.abs(zz))

sims = np.array([daily_pvalues() for _ in range(500)])
print(f'Falso positivo olhando só no fim (dia 30):   {(sims[:, -1] < 0.05).mean():.1%}')
print(f'Falso positivo parando no 1º dia com p<0,05: {(sims < 0.05).any(axis=1).mean():.1%}')

## 7. Decisão

Preencha com suas palavras, como faria para um stakeholder:

- **Hipótese:** …
- **Resultado da métrica principal:** lift, intervalo de confiança, p-valor
- **Métricas secundárias / de guarda:** …
- **Validade:** SRM ok? Duração respeitou o planejamento?
- **Decisão:** lançar / não lançar / estender, e por quê
- **Aprendizado:** o que o A/A e o peeking mostraram sobre o processo